# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Haryomhidhe/Machine-learning-flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one page on one specific day. I'm using the fact_content_daily_performance table, filtered to month=2026-02

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: total_impressions, avg_position, and sum_position, calculated from real GSC data in fact_content_daily_performance for February 2026

Label:-Opportunity score

context:- Content-ID, Client-ID.

Excluded:- Trend_direction and Trend_pct

trend_pct and trend_direction are excluded because they compare the first half of the month to the second half, meaning they require knowing what happens later in the month, information that wouldn't be available yet at the moment someone is actually deciding whether to refresh a page

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Query 1: The row count for February 2026 shows that every time a page gets checked on a given day, one row is created for that day. 7,355,108 is the total number of those page-day checks across the whole month.

Query 2: The grain check confirms how many times each page was checked across the whole month. With 321,546 unique pages and about 23 rows per page, this proves one row equals one page on one specific day, matching the original contract claim.

Query 3: This checks how many rows have real tracking data recorded, using a boolean column that's either true or false. Out of all the rows, 2,621,783 have real tracking data available, while the rest are missing it.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      7355108 │
└──────────────┘



In [24]:
con.sql(f"""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT content_hash_id || '_' || content_hash_id) AS unique_page_days
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────┐
│ total_rows │ unique_page_days │
│   int64    │      int64       │
├────────────┼──────────────────┤
│    7355108 │           321546 │
└────────────┴──────────────────┘



In [25]:
con.sql(f"""
SELECT COUNT(*) AS rows_with_gsc_data
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
WHERE gsc_data_available IS TRUE
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┐
│ rows_with_gsc_data │
│       int64        │
├────────────────────┤
│            2621783 │
└────────────────────┘



In [26]:
con.sql(f"""
DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet') LIMIT 1
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [27]:
con.sql(f"""
SELECT content_hash_id, report_date, COUNT(*) AS c
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id, report_date
HAVING c > 1
LIMIT 5
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬─────────────┬───────┐
│ content_hash_id │ report_date │   c   │
│     varchar     │    date     │ int64 │
├─────────────────┴─────────────┴───────┤
│                0 rows                 │
└───────────────────────────────────────┘



In [28]:
con.sql(f"""
SELECT MIN(report_date) AS earliest, MAX(report_date) AS latest
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
""").show()

┌────────────┬────────────┐
│  earliest  │   latest   │
│    date    │    date    │
├────────────┼────────────┤
│ 2026-02-01 │ 2026-02-28 │
└────────────┴────────────┘



In [29]:
con.sql(f"""
SELECT content_hash_id,
       SUM(CASE WHEN report_date <= '2026-02-14' THEN gsc_clicks ELSE 0 END) AS clicks_first_half,
       SUM(CASE WHEN report_date > '2026-02-14' THEN gsc_clicks ELSE 0 END) AS clicks_second_half
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
LIMIT 5
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬───────────────────┬────────────────────┐
│     content_hash_id      │ clicks_first_half │ clicks_second_half │
│         varchar          │      int128       │       int128       │
├──────────────────────────┼───────────────────┼────────────────────┤
│ content_1eea820697c3b95a │                 0 │                  0 │
│ content_9abd8b303f805847 │                 4 │                  2 │
│ content_5f58c55cbfee172a │                 0 │                  0 │
│ content_6fe390ba3af1e456 │                 0 │                  3 │
│ content_3ad5d2160242b9ca │                 0 │                  2 │
└──────────────────────────┴───────────────────┴────────────────────┘



In [30]:
con.sql(f"""
SELECT content_hash_id,
       SUM(CASE WHEN report_date <= '2026-02-14' THEN gsc_clicks ELSE 0 END) AS clicks_first_half,
       SUM(CASE WHEN report_date > '2026-02-14' THEN gsc_clicks ELSE 0 END) AS clicks_second_half,
       CASE
           WHEN SUM(CASE WHEN report_date <= '2026-02-14' THEN gsc_clicks ELSE 0 END) = 0 THEN NULL
           ELSE (SUM(CASE WHEN report_date > '2026-02-14' THEN gsc_clicks ELSE 0 END) - SUM(CASE WHEN report_date <= '2026-02-14' THEN gsc_clicks ELSE 0 END)) * 100.0
                / SUM(CASE WHEN report_date <= '2026-02-14' THEN gsc_clicks ELSE 0 END)
       END AS trend_pct,
       CASE
           WHEN SUM(CASE WHEN report_date > '2026-02-14' THEN gsc_clicks ELSE 0 END) > SUM(CASE WHEN report_date <= '2026-02-14' THEN gsc_clicks ELSE 0 END) THEN 'up'
           WHEN SUM(CASE WHEN report_date > '2026-02-14' THEN gsc_clicks ELSE 0 END) < SUM(CASE WHEN report_date <= '2026-02-14' THEN gsc_clicks ELSE 0 END) THEN 'down'
           ELSE 'flat'
       END AS trend_direction
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
LIMIT 5
""").show()

┌──────────────────────────┬───────────────────┬────────────────────┬───────────┬─────────────────┐
│     content_hash_id      │ clicks_first_half │ clicks_second_half │ trend_pct │ trend_direction │
│         varchar          │      int128       │       int128       │  double   │     varchar     │
├──────────────────────────┼───────────────────┼────────────────────┼───────────┼─────────────────┤
│ content_1eea820697c3b95a │                 0 │                  0 │      NULL │ flat            │
│ content_9abd8b303f805847 │                 4 │                  2 │     -50.0 │ down            │
│ content_5f58c55cbfee172a │                 0 │                  0 │      NULL │ flat            │
│ content_6fe390ba3af1e456 │                 0 │                  3 │      NULL │ up              │
│ content_3ad5d2160242b9ca │                 0 │                  2 │      NULL │ up              │
└──────────────────────────┴───────────────────┴────────────────────┴───────────┴─────────────────┘


In [31]:
df = con.sql(f"""
SELECT content_hash_id,
       SUM(gsc_impressions) AS total_impressions,
       SUM(gsc_clicks) AS total_clicks,
       AVG(gsc_avg_position) AS avg_position,
       SUM(gsc_sum_position) AS sum_position,
       CASE WHEN SUM(gsc_clicks) > 0 THEN 1 ELSE 0 END AS got_clicks
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,total_impressions,total_clicks,avg_position,sum_position,got_clicks
0,content_fb84747a57b8b665,0.0,0.0,NaN,0.0,0
1,content_feccf822ac21326e,0.0,0.0,NaN,0.0,0
2,content_17cf93c10413ebe9,0.0,0.0,NaN,0.0,0
3,content_a9905735266f8697,0.0,0.0,NaN,0.0,0
4,content_31c34765e7bba2f0,0.0,0.0,NaN,0.0,0


In [32]:
df['got_clicks'].value_counts()

,count
got_clicks,
0,266456
1,55090


In [33]:
df[['total_impressions', 'avg_position', 'sum_position', 'got_clicks']].corr()['got_clicks']

,got_clicks
total_impressions,0.387940
avg_position,-0.196119
sum_position,0.299774
got_clicks,1.000000


In [34]:
df_leak = con.sql(f"""
SELECT content_hash_id,
       SUM(gsc_impressions) AS total_impressions,
       AVG(gsc_avg_position) AS avg_position,
       SUM(gsc_sum_position) AS sum_position,
       CASE
           WHEN SUM(CASE WHEN report_date <= '2026-02-14' THEN gsc_clicks ELSE 0 END) = 0 THEN NULL
           ELSE (SUM(CASE WHEN report_date > '2026-02-14' THEN gsc_clicks ELSE 0 END) - SUM(CASE WHEN report_date <= '2026-02-14' THEN gsc_clicks ELSE 0 END)) * 100.0
                / SUM(CASE WHEN report_date <= '2026-02-14' THEN gsc_clicks ELSE 0 END)
       END AS trend_pct,
       CASE WHEN SUM(gsc_clicks) > 0 THEN 1 ELSE 0 END AS got_clicks
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY content_hash_id
""").df()

df_leak[['total_impressions', 'avg_position', 'sum_position', 'trend_pct', 'got_clicks']].corr()['got_clicks']

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,got_clicks
total_impressions,0.387940
avg_position,-0.196119
sum_position,0.299774
trend_pct,NaN
got_clicks,1.000000


In [35]:
result = con.sql("""
SELECT
  COUNT(*) as count,
  MIN(gsc_impressions) as min_imp,
  MAX(gsc_impressions) as max_imp,
  AVG(gsc_impressions) as avg_imp,
  PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY gsc_impressions) as p25_imp,
  PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY gsc_impressions) as p50_imp,
  PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY gsc_impressions) as p75_imp
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')
""").to_df()

result

,count,min_imp,max_imp,avg_imp,p25_imp,p50_imp,p75_imp
0,7355108,0,74688,24.801403,0.0,0.0,5.0


In [36]:
result_pos = con.sql("""
SELECT
  COUNT(*) as count,
  MIN(gsc_avg_position) as min_pos,
  MAX(gsc_avg_position) as max_pos,
  AVG(gsc_avg_position) as avg_pos,
  PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY gsc_avg_position) as p25_pos,
  PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY gsc_avg_position) as p50_pos,
  PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY gsc_avg_position) as p75_pos
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')
""").to_df()

result_pos

,count,min_pos,max_pos,avg_pos,p25_pos,p50_pos,p75_pos
0,7355108,0.0,633.0,12.09891,3.142857,6.666667,13.580645


In [37]:
result_ctr = con.sql("""
SELECT
  COUNT(*) as count,
  MIN(CASE WHEN gsc_impressions > 0 THEN (gsc_clicks::FLOAT / gsc_impressions) * 100 ELSE 0 END) as min_ctr,
  MAX(CASE WHEN gsc_impressions > 0 THEN (gsc_clicks::FLOAT / gsc_impressions) * 100 ELSE 0 END) as max_ctr,
  AVG(CASE WHEN gsc_impressions > 0 THEN (gsc_clicks::FLOAT / gsc_impressions) * 100 ELSE 0 END) as avg_ctr,
  PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY CASE WHEN gsc_impressions > 0 THEN (gsc_clicks::FLOAT / gsc_impressions) * 100 ELSE 0 END) as p25_ctr,
  PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY CASE WHEN gsc_impressions > 0 THEN (gsc_clicks::FLOAT / gsc_impressions) * 100 ELSE 0 END) as p50_ctr,
  PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY CASE WHEN gsc_impressions > 0 THEN (gsc_clicks::FLOAT / gsc_impressions) * 100 ELSE 0 END) as p75_ctr
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')
""").to_df()

result_ctr

,count,min_ctr,max_ctr,avg_ctr,p25_ctr,p50_ctr,p75_ctr
0,7355108,0.0,100.0,0.113302,0.0,0.0,0.0


In [38]:
result_clicks = con.sql("""
SELECT
  COUNT(*) as count,
  MIN(gsc_clicks) as min_clicks,
  MAX(gsc_clicks) as max_clicks,
  AVG(gsc_clicks) as avg_clicks,
  PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY gsc_clicks) as p25_clicks,
  PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY gsc_clicks) as p50_clicks,
  PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY gsc_clicks) as p75_clicks
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')
""").to_df()

result_clicks

,count,min_clicks,max_clicks,avg_clicks,p25_clicks,p50_clicks,p75_clicks
0,7355108,0,175,0.080713,0.0,0.0,0.0


In [39]:
result = con.sql("""
SELECT
  CASE
    WHEN gsc_impressions = 0 THEN '0_none'
    WHEN gsc_impressions <= 5 THEN '1_low'
    WHEN gsc_impressions <= 100 THEN '2_medium'
    ELSE '3_high'
  END as impression_bucket,
  COUNT(*) as n,
  AVG(CASE WHEN gsc_impressions > 0 THEN (gsc_clicks::FLOAT / gsc_impressions) * 100 ELSE NULL END) as avg_ctr
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')
GROUP BY impression_bucket
ORDER BY impression_bucket
""").to_df()

result

,impression_bucket,n,avg_ctr
0,0_none,4641069,NaN
1,1_low,813201,0.415065
2,2_medium,1382891,0.256554
3,3_high,517947,0.331300


In [40]:
result2 = con.sql("""
SELECT
  trend_direction,
  COUNT(*) as n,
  AVG(gsc_impressions) as avg_impressions,
  AVG(CASE WHEN gsc_impressions > 0 THEN (gsc_clicks::FLOAT / gsc_impressions) * 100 ELSE NULL END) as avg_ctr
FROM (
  SELECT
    content_hash_id,
    SUM(gsc_impressions) as gsc_impressions,
    SUM(gsc_clicks) as gsc_clicks,
    CASE
      WHEN SUM(CASE WHEN report_date > '2026-02-14' THEN gsc_clicks ELSE 0 END) >
           SUM(CASE WHEN report_date <= '2026-02-14' THEN gsc_clicks ELSE 0 END) THEN 'up'
      WHEN SUM(CASE WHEN report_date > '2026-02-14' THEN gsc_clicks ELSE 0 END) <
           SUM(CASE WHEN report_date <= '2026-02-14' THEN gsc_clicks ELSE 0 END) THEN 'down'
      ELSE 'flat'
    END as trend_direction
  FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet')
  GROUP BY content_hash_id
)
GROUP BY trend_direction
""").to_df()

result2

,trend_direction,n,avg_impressions,avg_ctr
0,up,29855,2868.429107,1.458394
1,flat,272729,110.834238,0.037116
2,down,18962,3389.107689,1.518540


When I added trend_pct to test for leakage, the correlation came back as NaN instead of a clean high number. This happened because trend_pct can only be calculated for pages that already had clicks in the first half of the month, which means got_clicks is always 1 wherever trend_pct exists, so there's no real variation left to measure. This shows trend_pct isn't just a strong predictor, it's mathematically tied to the answer itself, which is why it has to be excluded. My honest baseline, using only total_impressions, avg_position, and sum_position, gave correlations around 0.30 to 0.39, and that's the real, trustworthy result I'm keeping.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can't tell me which pages people actually liked or found satisfying, it only shows that people engaged with a page, not whether they were happy with it. It also can't tell me anything meaningful about the roughly 4.7 million rows missing GSC tracking data, missing data doesn't mean zero performance, it just means no reliable signal exists for those pages at all.

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df = con.sql(f"""
SELECT content_hash_id, gsc_impressions, gsc_clicks, gsc_avg_position, gsc_sum_position,
       CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions ELSE 0 END AS gsc_ctr
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-02/*.parquet')
LIMIT 10
""").df()

df.head()

,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_sum_position,gsc_ctr
0,content_7995404695ee1ffd,57,0,31.192982,1778,0.0
1,content_1eea820697c3b95a,13,0,6.538462,85,0.0
2,content_ccbb253f142217c3,59,0,16.966102,1001,0.0
3,content_ae16a6b9cf64c80a,17,0,16.882353,287,0.0
4,content_acf700633f016e5a,6,0,4.500000,27,0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.